# 🧹 Notebook 02 — Data Cleaning & Validation
**Urban Taxi Demand Pattern Analysis**

This notebook loads raw Parquet files, applies quality checks, removes invalid data, engineers new features, and saves cleaned files to `data/processed/`.

---

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
from pathlib import Path

from utils import (
    DATA_RAW, DATA_PROCESSED,
    log_row_counts, winsorize_col, load_raw
)

## 1. Configuration

In [ ]:
TAXI_TYPES = ['yellow', 'green']
YEAR = 2024
MONTHS = list(range(1, 7))  # Must match Notebook 01

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
print(f"Processing {len(TAXI_TYPES) * len(MONTHS)} files")

## 2. Cleaning Pipeline Functions

In [ ]:
def standardize_columns(df: pd.DataFrame, taxi_type: str) -> pd.DataFrame:
    """Rename pickup/dropoff columns to a common format."""
    if taxi_type == 'yellow':
        df = df.rename(columns={
            'tpep_pickup_datetime': 'pickup_datetime',
            'tpep_dropoff_datetime': 'dropoff_datetime'
        })
    elif taxi_type == 'green':
        df = df.rename(columns={
            'lpep_pickup_datetime': 'pickup_datetime',
            'lpep_dropoff_datetime': 'dropoff_datetime'
        })
    df['taxi_type'] = taxi_type
    return df


def drop_nulls_on_key_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Drop rows missing critical fields."""
    key_cols = ['pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID']
    before = len(df)
    df = df.dropna(subset=key_cols)
    log_row_counts(before, len(df), 'Drop nulls on key cols')
    return df


def filter_invalid_timestamps(df: pd.DataFrame) -> pd.DataFrame:
    """Remove trips where dropoff is before or equal to pickup."""
    before = len(df)
    df = df[df['dropoff_datetime'] > df['pickup_datetime']]
    log_row_counts(before, len(df), 'Invalid timestamps')
    return df


def remove_negative_values(df: pd.DataFrame) -> pd.DataFrame:
    """Remove rows with negative fares or distances."""
    before = len(df)
    df = df[(df['trip_distance'] > 0) & (df['fare_amount'] >= 0)]
    log_row_counts(before, len(df), 'Negative fares/distances')
    return df


def filter_passenger_count(df: pd.DataFrame) -> pd.DataFrame:
    """Keep only trips with 1–6 passengers."""
    before = len(df)
    if 'passenger_count' in df.columns:
        df = df[(df['passenger_count'] >= 1) & (df['passenger_count'] <= 6)]
    log_row_counts(before, len(df), 'Passenger count filter')
    return df


def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    """Engineer new columns from existing data."""
    # Trip duration in minutes
    df['trip_duration_min'] = (
        (df['dropoff_datetime'] - df['pickup_datetime']).dt.total_seconds() / 60
    )

    # Time-based features
    df['pickup_hour'] = df['pickup_datetime'].dt.hour
    df['pickup_day_of_week'] = df['pickup_datetime'].dt.dayofweek  # 0=Monday
    df['pickup_date'] = df['pickup_datetime'].dt.date
    df['pickup_month'] = df['pickup_datetime'].dt.month

    # Speed (miles per hour)
    df['speed_mph'] = np.where(
        df['trip_duration_min'] > 0,
        df['trip_distance'] / (df['trip_duration_min'] / 60),
        np.nan
    )

    return df


def filter_extreme_trips(df: pd.DataFrame) -> pd.DataFrame:
    """Remove trips that are unreasonably long (> 180 min) or fast (> 100 mph)."""
    before = len(df)
    df = df[df['trip_duration_min'] <= 180]
    df = df[(df['speed_mph'].isna()) | (df['speed_mph'] <= 100)]
    log_row_counts(before, len(df), 'Extreme trip filter')
    return df

## 3. Run the Cleaning Pipeline

In [ ]:
cleaning_summary = []

for taxi_type in TAXI_TYPES:
    for month in MONTHS:
        filename = f"{taxi_type}_tripdata_{YEAR}-{month:02d}.parquet"
        raw_path = DATA_RAW / filename

        if not raw_path.exists():
            print(f"⏭️  Skipping (not found): {filename}")
            continue

        print(f"\n{'='*60}")
        print(f"Processing: {filename}")
        print('='*60)

        df = pd.read_parquet(raw_path)
        initial_rows = len(df)

        # Pipeline steps
        df = standardize_columns(df, taxi_type)
        df = drop_nulls_on_key_cols(df)
        df = filter_invalid_timestamps(df)
        df = remove_negative_values(df)
        df = filter_passenger_count(df)
        df = add_derived_features(df)
        df = filter_extreme_trips(df)

        # Winsorize key numeric columns
        for col in ['trip_distance', 'fare_amount', 'total_amount']:
            if col in df.columns:
                df = winsorize_col(df, col, lower=0.01, upper=0.99)

        # Guard: if everything was dropped, something is very wrong
        if len(df) == 0:
            raise ValueError(f"All rows dropped for {filename}! Check cleaning rules.")

        # Save cleaned file
        out_name = f"cleaned_{taxi_type}_{YEAR}-{month:02d}.parquet"
        out_path = DATA_PROCESSED / out_name
        df.to_parquet(out_path, index=False)

        final_rows = len(df)
        pct_kept = (final_rows / initial_rows * 100) if initial_rows > 0 else 0
        print(f"\n  ✅ Saved: {out_name}")
        print(f"  📊 Kept {final_rows:,} / {initial_rows:,} rows ({pct_kept:.1f}%)")

        cleaning_summary.append({
            'file': filename,
            'raw_rows': initial_rows,
            'clean_rows': final_rows,
            'pct_kept': round(pct_kept, 1)
        })

## 4. Cleaning Summary

In [ ]:
if cleaning_summary:
    summary_df = pd.DataFrame(cleaning_summary)
    print("\n📋 Cleaning Summary")
    print("=" * 60)
    display(summary_df)

    total_raw = summary_df['raw_rows'].sum()
    total_clean = summary_df['clean_rows'].sum()
    print(f"\nTotal: {total_clean:,} / {total_raw:,} rows kept "
          f"({total_clean/total_raw*100:.1f}%)")
else:
    print("No files were processed. Run Notebook 01 first.")

## 5. Quick Preview of Cleaned Data

In [ ]:
# Preview first cleaned file
sample = DATA_PROCESSED / f"cleaned_yellow_{YEAR}-{MONTHS[0]:02d}.parquet"
if sample.exists():
    df_peek = pd.read_parquet(sample)
    print(f"Shape: {df_peek.shape}")
    print(f"\nColumns: {list(df_peek.columns)}")
    print(f"\nDtypes:\n{df_peek.dtypes}")
    df_peek.head()

---
**Next →** `03_eda.ipynb`